# E5 — Demo: Pipeline completo E2→E3→E4

Notebook de demostración end-to-end del pipeline de reparación 3D:

```
[Vóxeles E2]  ──►  convertir_voxels_a_nube  ──►  [Nube rota (2048,3)]
                                                         │
                                                         ▼
                                              PoinTr  (E3 shape completion)
                                                         │
                                                         ▼
                                             [Nube completa (2048+N,3)]
                                                         │
                                                         ▼
                                        Poisson + manifold3d  (E4)
                                                         │
                                                         ▼
                                                 [STL imprimible]
```

**Modo de uso:**
- **Con vóxeles reales de E2**: sube el fichero `.npy` de salida de Pix2Vox++ → la Celda 3 lo convierte.
- **Sin E2 todavía** (demo): la Celda 3 usa una nube de puntos sintética del dataset de test.

Al final, la Celda 9 lanza una **aplicación Gradio** accesible desde el navegador.

---
## Sección 1 — Instalación y clonado

In [ ]:
import subprocess, os
from getpass import getpass

for pkg in ['trimesh', 'manifold3d', 'pymeshlab', 'plotly', 'gradio',
            'scipy', 'numpy', 'easydict', 'timm']:
    r = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN"}] {pkg}')

# Clonar repos
if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')

REPO = '/content/TFM'
if not os.path.exists(REPO):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO,'-q'], capture_output=True)
    del token; print('Repo clonado.')
else:
    subprocess.run(['git','-C',REPO,'pull','-q'], capture_output=True)
    print('[OK] Repo actualizado.')

os.chdir(REPO)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)
print('[OK] listo')

---
## Sección 2 — Cargar modelo PoinTr (E3)

In [ ]:
import sys, types, glob as _glob, importlib
import torch, torch.nn as nn, numpy as np
from pathlib import Path
from easydict import EasyDict
from google.colab import drive

if not (Path('/content/drive').exists() and list(Path('/content/drive').iterdir())):
    drive.mount('/content/drive')
    print('Drive montado.')

DRIVE = '/content/drive/MyDrive'
VERSION_E3 = 'v6_obj_sn'   # modelo PoinTr a usar
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

# ── Sys path ─────────────────────────────────────────────────
for p in ['/content/TFM', '/content/PoinTr']:
    if p in sys.path: sys.path.remove(p)
sys.path.insert(0, '/content/TFM')
sys.path.insert(0, '/content/PoinTr')
importlib.invalidate_caches()
os.chdir('/content/TFM')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')

# ── Mocks CUDA (igual que en entrenamiento) ───────────────────
_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]
importlib.invalidate_caches()

for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
        if new!=src: open(fp,'w',encoding='utf-8').write(new)
    except: pass

def _force(n,a):
    m=types.ModuleType(n)
    for k,v in a.items(): setattr(m,k,v)
    sys.modules[n]=m

def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
class _CL1(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
_ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL1,'ChamferDistanceL1_PM':_CL1,'chamfer_3DDist':_cr}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']:
    _force(n,_ch)

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np_):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np_,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np_):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1)
            dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu

class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})

def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']:
        if pfx+base not in sys.modules: _force(pfx+base,attrs)

# ── Cargar checkpoint ─────────────────────────────────────────
ckpt_local = f'E3/checkpoints_pointr_{VERSION_E3}/best.pt'
ckpt_drive  = f'{BASE_E3}/modelos/{VERSION_E3}/best.pt'
ckpt_path   = ckpt_local if Path(ckpt_local).exists() else ckpt_drive
ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'PoinTr {VERSION_E3} — best epoch {ck["epoch"]}')

try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict'])
model = model.to(device).eval()
print(f'Modelo listo. Params: {sum(p.numel() for p in model.parameters()):,}')

---
## Sección 3 — Cargar nube rota (desde E2 o demo sintética)

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPCION A — Vóxeles de E2 (Pix2Vox++): subir fichero .npy
# ══════════════════════════════════════════════════════════════
USAR_VOXELES_E2 = False  # ← cambia a True si tienes salida de Pix2Vox++

if USAR_VOXELES_E2:
    from google.colab import files
    print('Sube el fichero .npy de vóxeles de E2 (shape 32×32×32):')
    subidos = files.upload()
    ruta_voxels = list(subidos.keys())[0]

    # Conversión E2 → E3 usando el script del proyecto
    import sys; sys.path.insert(0, '/content/TFM')
    from Scripts.convertir_voxels_a_nube import voxels_a_nube, diagnosticar_voxels

    voxels = np.load(ruta_voxels)
    diag = diagnosticar_voxels(voxels)
    print(f'Grid: {diag["forma_grid"]}  ocupado: {diag["porcentaje_ocupado"]}%')
    for w in diag['advertencias']:
        if w: print(f'  ⚠️  {w}')

    nube_rota = voxels_a_nube(voxels, n_puntos=2048)
    NOMBRE_OBJETO = Path(ruta_voxels).stem
    print(f'Nube generada desde vóxeles: {nube_rota.shape}  radio_max={np.linalg.norm(nube_rota,axis=1).max():.3f}')

else:
    # ══════════════════════════════════════════════════════════
    # OPCION B — Demo: usar una nube del test set de E3
    # ══════════════════════════════════════════════════════════
    import sys; sys.path.insert(0, '/content/TFM')
    import E3.dataset as _ds; _ds.CENTRAR_EN_ROTO = False
    from E3.dataset import construir_pares
    import random

    DRIVE = '/content/drive/MyDrive'
    BASE_GEN = f'{DRIVE}/Datos_E2_E3/General'
    carpetas = [f'{BASE_GEN}/shapenet_roturas', f'{BASE_GEN}/roturas_Objaverse_v2']
    carpetas_ok = [c for c in carpetas if Path(c).exists()]

    if not carpetas_ok:
        # Fallback: usar test set local si existe
        carpetas_ok = [c for c in ['Datos/shapenet/roturas', 'Datos/objaverse/roturas_v2']
                       if Path(c).exists()]

    todos = construir_pares(carpetas_ok)
    rng = random.Random(42); rng.shuffle(todos)
    n = len(todos); _te = todos[int(0.9*n):]

    # Elegir el mejor objeto del test (el que mejor sale con PoinTr)
    # Por defecto: primero del test set
    INDICE_DEMO = 0   # ← cambia para ver otros objetos
    ruta_roto, ruta_comp = _te[INDICE_DEMO]
    nube_rota = np.load(ruta_roto).astype(np.float32)
    nube_gt   = np.load(ruta_comp).astype(np.float32)
    NOMBRE_OBJETO = Path(ruta_roto).stem.replace('_roto','')
    print(f'Objeto de demo: {NOMBRE_OBJETO}')
    print(f'Nube rota: {nube_rota.shape}  radio_max={np.linalg.norm(nube_rota,axis=1).max():.3f}')
    print(f'(GT disponible para comparación — no se usa como entrada al modelo)')

---
## Sección 4 — E3: PoinTr shape completion

In [ ]:
def inferir_pointr(nube_np: np.ndarray) -> np.ndarray:
    """Nube rota (2048,3) → nube completa predicha (N,3)."""
    with torch.no_grad():
        inp = torch.tensor(nube_np, dtype=torch.float32).unsqueeze(0).to(device)
        out = model(inp)
        fine = out[-1] if isinstance(out, (list, tuple)) else out
        return fine.squeeze(0).cpu().numpy()

def filtrar_outliers(pts, k=20, std_ratio=1.5):
    from scipy.spatial import cKDTree
    tree = cKDTree(pts)
    dists, _ = tree.query(pts, k=k+1)
    mean_d = dists[:, 1:].mean(axis=1)
    umbral = mean_d.mean() + std_ratio * mean_d.std()
    mask = mean_d < umbral
    return pts[mask], int((~mask).sum())

def chamfer_l1(p, g):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    d = torch.cdist(pt, gt, p=2)
    return ((d.min(2).values.mean() + d.min(1).values.mean()) / 2).item()

# Inferencia E3
print('Ejecutando PoinTr...')
pred_raw = inferir_pointr(nube_rota)
pred, n_outliers = filtrar_outliers(pred_raw)
print(f'PoinTr: {pred_raw.shape[0]} pts → {pred.shape[0]} pts (eliminados {n_outliers} outliers)')

# Métricas (solo si tenemos GT)
if 'nube_gt' in dir() or 'nube_gt' in locals():
    cd_val = chamfer_l1(pred, nube_gt)
    print(f'CD-L1 vs GT: {cd_val:.4f}')

# Visualización de la nube completa predicha
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pred[:,0], y=pred[:,1], z=pred[:,2], mode='markers',
    name='PoinTr (completo)', marker=dict(size=2, color='#66BB6A', opacity=0.85)))
fig.add_trace(go.Scatter3d(x=nube_rota[:,0], y=nube_rota[:,1], z=nube_rota[:,2], mode='markers',
    name='Roto (entrada)', marker=dict(size=3, color='#EF5350', opacity=0.95)))
fig.update_layout(
    scene=dict(bgcolor='#111',
               xaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               yaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               zaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               aspectmode='cube'),
    title=dict(text=f'<b>E3: Shape completion</b> — {NOMBRE_OBJETO}<br><sup>Rojo=roto | Verde=reconstruido por PoinTr</sup>',
               font=dict(color='white'), x=0.5),
    paper_bgcolor='#111', legend=dict(font=dict(color='white')),
    height=560, width=680, margin=dict(l=0,r=0,t=60,b=0))
fig.show()
print('E3 completado.')

---
## Sección 5 — E4: Generación de STL

In [ ]:
import pymeshlab, trimesh
from scipy.spatial import cKDTree

TAMANO_MM = 100.0

def suavizar_nube(pts, k=15, iters=3):
    tree = cKDTree(pts)
    _, idx = tree.query(pts, k=k+1)
    result = pts.copy()
    for _ in range(iters):
        result = result[idx[:, 1:]].mean(axis=1)
    return result.astype(np.float32)

def poisson_stl(pts, depth=7):
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pts.astype(np.float64)))
    ms.compute_normal_for_point_clouds(k=20, smoothiter=2)
    ms.generate_surface_reconstruction_screened_poisson(depth=depth, scale=1.1)
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=200)
    ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)
    m = ms.current_mesh()
    return trimesh.Trimesh(vertices=m.vertex_matrix(), faces=m.face_matrix(), process=False)

def reparar(mesh):
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda c: len(c.faces))
    try:
        import manifold3d
        m = manifold3d.Manifold(manifold3d.Mesh(
            vert_properties=np.array(mesh.vertices, dtype=np.float32),
            tri_verts=np.array(mesh.faces, dtype=np.uint32)))
        out = m.to_mesh()
        res = trimesh.Trimesh(vertices=np.array(out.vert_properties),
                              faces=np.array(out.tri_verts), process=False)
        if len(res.vertices) > 0:
            mesh = res
    except: pass
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    mesh.process(validate=False)
    return mesh

# Pipeline E4
print('Ejecutando E4...')
pred_suav = suavizar_nube(pred, k=15, iters=3)
print(f'  Suavizado Laplaciano: {pred.shape[0]} pts')

mesh_raw = poisson_stl(pred_suav, depth=7)
print(f'  Poisson depth=7: {len(mesh_raw.faces):,} caras')

mesh_raw.apply_translation(-mesh_raw.centroid)
lado = mesh_raw.bounding_box.extents.max()
if lado > 0: mesh_raw.apply_scale(TAMANO_MM / lado)

mesh_final = reparar(mesh_raw)
wt = bool(mesh_final.is_watertight)
eu = int(mesh_final.euler_number)
print(f'  Reparación: {len(mesh_final.faces):,} caras | watertight={wt} | euler={eu}')
print()
print('═'*50)
print(f'RESULTADO: {"✅ WATERTIGHT" if wt else "⚠️  No watertight"}')
print(f'  Caras   : {len(mesh_final.faces):,}')
print(f'  Euler   : {eu}  (0=esfera, ±2=taza con/sin asa)')
print(f'  Tamaño  : {TAMANO_MM:.0f} mm lado mayor')
print('═'*50)

# Guardar STL
Path('E5').mkdir(exist_ok=True)
ruta_stl = f'E5/{NOMBRE_OBJETO}_demo.stl'
mesh_final.export(ruta_stl)
print(f'STL guardado: {ruta_stl}')

# Visualización STL
verts = np.array(mesh_final.vertices)
faces = np.array(mesh_final.faces)
if len(faces) > 0:
    z = verts[:,2]
    intensidad = (z - z.min()) / (np.ptp(z) + 1e-8)
    fig2 = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#0D47A1'],[0.5,'#29B6F6'],[1,'#E1F5FE']],
        showscale=False,
        lighting=dict(ambient=0.3,diffuse=0.85,roughness=0.3,specular=0.6),
        lightposition=dict(x=200,y=300,z=400)))
    fig2.update_layout(
        scene=dict(bgcolor='#111',
                   xaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   yaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   zaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   aspectmode='data'),
        title=dict(text=f'<b>E4: STL final</b> — {NOMBRE_OBJETO}<br>'
                        f'<sup>{"WATERTIGHT" if wt else "no watertight"} | euler={eu} | {len(faces):,} caras</sup>',
                   font=dict(color='white'), x=0.5),
        paper_bgcolor='#111', height=560, width=680,
        margin=dict(l=0,r=0,t=60,b=0))
    fig2.show()

---
## Sección 6 — Descargar STL

In [ ]:
from google.colab import files
if Path(ruta_stl).exists():
    print(f'Descargando {ruta_stl}...')
    files.download(ruta_stl)
else:
    print('STL no encontrado — ejecuta la Sección 5 primero.')

---
## Sección 7 — App Gradio (interfaz web)

Lanza una interfaz web accesible desde el navegador.

**Entradas:**
- `.npy` de vóxeles (salida de E2 / Pix2Vox++) — se convierte automáticamente
- `.npy` de nube de puntos (2048,3) — se usa directamente como entrada a E3

**Salida:** STL descargable

> Si Colab muestra un enlace `*.gradio.live`, ábrelo en cualquier navegador.

In [ ]:
import gradio as gr
import tempfile, numpy as np, torch
from pathlib import Path

# Importar conversor E2→E3
sys.path.insert(0, '/content/TFM')
from Scripts.convertir_voxels_a_nube import voxels_a_nube, diagnosticar_voxels

def detectar_tipo_npy(arr):
    """Decide si el .npy es vóxeles (3D+) o nube de puntos (N,3)."""
    while arr.ndim > 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim == 3:               # grid de vóxeles
        return 'voxels'
    if arr.ndim == 2 and arr.shape[1] == 3:  # nube de puntos
        return 'nube'
    return 'desconocido'

def pipeline_completo(archivo_npy):
    """Función principal del pipeline E2→E3→E4."""
    if archivo_npy is None:
        return None, "⚠️ Sube un fichero .npy primero"

    try:
        arr = np.load(archivo_npy.name).astype(np.float32)
        tipo = detectar_tipo_npy(arr)
        log = [f'Archivo: {Path(archivo_npy.name).name}  shape={arr.shape}  tipo={tipo}']

        # ── E2 → E3: conversión si son vóxeles ───────────────────────────────
        if tipo == 'voxels':
            diag = diagnosticar_voxels(arr)
            log.append(f'Grid: {diag["forma_grid"]}  ocupado: {diag["porcentaje_ocupado"]}%')
            for w in diag['advertencias']:
                if w: log.append(f'  ⚠️ {w}')
            nube_rota = voxels_a_nube(arr, n_puntos=2048)
            log.append(f'Convertido a nube: {nube_rota.shape}')
        elif tipo == 'nube':
            nube_rota = arr
            if len(nube_rota) > 2048:
                idx = np.random.choice(len(nube_rota), 2048, replace=False)
                nube_rota = nube_rota[idx]
            elif len(nube_rota) < 512:
                return None, 'La nube tiene menos de 512 puntos — demasiado escasa'
            # Normalizar
            nube_rota -= nube_rota.mean(0)
            r = np.linalg.norm(nube_rota, axis=1).max()
            if r > 1e-8: nube_rota /= r
            log.append(f'Nube de puntos: {nube_rota.shape}')
        else:
            return None, f'Formato no reconocido: shape={arr.shape}'

        # ── E3: PoinTr shape completion ───────────────────────────────────────
        log.append('\nE3: PoinTr shape completion...')
        pred_raw = inferir_pointr(nube_rota)
        pred, n_out = filtrar_outliers(pred_raw)
        log.append(f'  Salida: {pred.shape[0]} pts (eliminados {n_out} outliers)')

        # ── E4: STL ───────────────────────────────────────────────────────────
        log.append('\nE4: Generación de STL...')
        pred_suav = suavizar_nube(pred)
        mesh_raw = poisson_stl(pred_suav, depth=7)
        log.append(f'  Poisson: {len(mesh_raw.faces):,} caras')
        mesh_raw.apply_translation(-mesh_raw.centroid)
        lado = mesh_raw.bounding_box.extents.max()
        if lado > 0: mesh_raw.apply_scale(100.0 / lado)
        mesh_final = reparar(mesh_raw)
        wt = bool(mesh_final.is_watertight)
        eu = int(mesh_final.euler_number)
        log.append(f'  Final: {len(mesh_final.faces):,} caras | watertight={wt} | euler={eu}')

        # ── Guardar STL temporal ──────────────────────────────────────────────
        nombre = Path(archivo_npy.name).stem
        tmp = tempfile.NamedTemporaryFile(suffix='.stl', delete=False,
                                          prefix=f'{nombre}_reparado_')
        mesh_final.export(tmp.name)
        log.append(f'\nSTL guardado: {Path(tmp.name).name}')
        estado = f'{"✅ WATERTIGHT" if wt else "⚠️  No watertight (procesable en Cura/PrusaSlicer)"}\n'
        estado += f'Euler={eu}  |  {len(mesh_final.faces):,} caras  |  100 mm lado mayor'
        return tmp.name, '\n'.join(log) + '\n\n' + estado

    except Exception as e:
        import traceback
        return None, f'Error: {e}\n\n{traceback.format_exc()}'


# ── Interfaz Gradio ───────────────────────────────────────────────────────────
with gr.Blocks(title='Pipeline 3D — TFM UCM', theme=gr.themes.Soft()) as app:

    gr.Markdown("""
# 🔧 Reparación 3D de objetos rotos
**Pipeline:** Vóxeles E2 (Pix2Vox++) → PoinTr shape completion → STL imprimible

Sube un fichero `.npy` con la salida de Pix2Vox++ (grid de vóxeles 32×32×32)
o una nube de puntos (N, 3). El sistema completa la geometría y genera un STL.
""")

    with gr.Row():
        with gr.Column(scale=1):
            entrada = gr.File(
                label='Fichero .npy (vóxeles de E2 o nube de puntos)',
                file_types=['.npy']
            )
            btn = gr.Button('▶ Reconstruir y generar STL', variant='primary')
            gr.Markdown("""
**Formatos aceptados:**
- `(D, H, W)` o `(1, D, H, W)` → grid de vóxeles de E2
- `(N, 3)` → nube de puntos directa

**Tiempo estimado:** 5-15 s (GPU T4 en Colab)
""")

        with gr.Column(scale=2):
            salida_stl = gr.File(label='STL generado (descarga)')
            salida_log = gr.Textbox(
                label='Log del pipeline',
                lines=18, interactive=False
            )

    btn.click(
        fn=pipeline_completo,
        inputs=[entrada],
        outputs=[salida_stl, salida_log]
    )

    gr.Examples(
        examples=[],  # Se añadirían ejemplos con ficheros .npy del test set
        inputs=[entrada],
        label='Ejemplos (requiere ficheros .npy en el servidor)'
    )

app.launch(share=True, debug=False)
print('App Gradio lanzada — abre el enlace de arriba en el navegador.')